# ML-04 — Search Intelligence Data Contract

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
from google.colab import userdata
from huggingface_hub import login

hf_token = userdata.get('HF_TOKEN')
login(token=hf_token)

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

We have one row per content for each date on the month of march

In [2]:
from huggingface_hub import HfApi

api = HfApi()
files = api.list_repo_files("FlyRank/internship-warehouse", repo_type="dataset")

relevant = [f for f in files if "fact_content_daily_performance" in f]
for f in relevant:
    print(f)

fact_content_daily_performance/month=2025-01/data_0.parquet
fact_content_daily_performance/month=2025-02/data_0.parquet
fact_content_daily_performance/month=2025-03/data_0.parquet
fact_content_daily_performance/month=2025-04/data_0.parquet
fact_content_daily_performance/month=2025-05/data_0.parquet
fact_content_daily_performance/month=2025-06/data_0.parquet
fact_content_daily_performance/month=2025-07/data_0.parquet
fact_content_daily_performance/month=2025-08/data_0.parquet
fact_content_daily_performance/month=2025-09/data_0.parquet
fact_content_daily_performance/month=2025-10/data_0.parquet
fact_content_daily_performance/month=2025-11/data_0.parquet
fact_content_daily_performance/month=2025-12/data_0.parquet
fact_content_daily_performance/month=2026-01/data_0.parquet
fact_content_daily_performance/month=2026-02/data_0.parquet
fact_content_daily_performance/month=2026-03/data_0.parquet
fact_content_daily_performance/month=2026-04/data_0.parquet
fact_content_daily_performance/month=202

In [3]:
from datasets import load_dataset

ds_march = load_dataset(
    "FlyRank/internship-warehouse",
    data_files="fact_content_daily_performance/month=2026-03/data_0.parquet",
    split="train",
    token=hf_token
)

df_march = ds_march.to_pandas()
print(df_march.shape)
print(df_march['report_date'].min(), df_march['report_date'].max())

df_march_slim = df_march[['report_date', 'content_hash_id']]

daily_check = df_march_slim.groupby('report_date')['content_hash_id'].agg(
    total_rows='count',
    unique_ids='nunique'
).reset_index()

mismatches = daily_check[daily_check['total_rows'] != daily_check['unique_ids']]
print(f"Dates with duplicate content_hash_id: {len(mismatches)}")

print("Therefore we have one row per content(page), for each date on 1st March 2026 to 1st March 2031")



(9841378, 30)
2026-03-01 2026-03-31
Dates with duplicate content_hash_id: 0
Therefore we have one row per content(page), for each date on 1st March 2026 to 1st March 2031


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

In [4]:
from huggingface_hub import HfApi

api = HfApi(token=hf_token)
files = api.list_repo_files("FlyRank/internship-warehouse", repo_type="dataset")

dim_files = [f for f in files if "dim_content" in f]
for f in dim_files:
    print(f)

dim_content.parquet


In [5]:
ds_content = load_dataset(
    "FlyRank/internship-warehouse",
    data_files="dim_content.parquet",
    split="train",
    token=hf_token
)

df_content = ds_content.to_pandas()


In [6]:
df_march_joined = df_march.merge(
    df_content,
    on='content_hash_id',
    how='left',
    validate='m:1'
)

In [ ]:
print("filter using TRUE for either of them and both")
df_ga4_gsc_march = df_march_joined[(df_march_joined['gsc_data_available'] == True)& (df_march_joined['ga4_data_available'] == True)]
df_ga4_march = df_march_joined[df_march_joined['ga4_data_available'] == True]
df_gsc_march = df_march_joined[df_march_joined['gsc_data_available'] == True].copy()
print("both ga4 and gsc availability",(df_ga4_gsc_march.shape[0]/ df_march_joined.shape[0]) * 100)
print("ga4 availability",(df_ga4_march.shape[0]/ df_march_joined.shape[0]) * 100)
print("gsc avialability", (df_gsc_march.shape[0]/ df_march_joined.shape[0]) * 100)



filter using TRUE for either of them and both


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.